In [1]:
import os
import re
import pandas as pd
import spacy
from tqdm import tqdm

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

print("Libraries loaded")

Libraries loaded


In [2]:
df = pd.read_csv("../data/processed/comments_clean.csv")

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head(3)

Shape: (27330, 8)
Columns: ['video_id', 'video_name', 'comment_id', 'author', 'text', 'likes', 'published_at', 'reply_count']


,video_id,video_name,comment_id,author,text,likes,published_at,reply_count
0,0iT9HbaRwfM,ai_slop_renaissance,Ugz2NWz28ZOlCRJLTZN4AaABAg,@AfterSkool,Wow. It blows my mind the amount of commenters...,5820,2026-03-18T19:38:57Z,164
1,0iT9HbaRwfM,ai_slop_renaissance,UgwP37nnkq5FwhBtw2N4AaABAg,@thearquitec2,you can't create without destroying because yo...,0,2026-05-31T01:01:28Z,0
2,0iT9HbaRwfM,ai_slop_renaissance,UgwzfIIcu4vxrcSXCdB4AaABAg,@angelraulrecioguerrero9969,I'm madly in love of the way you are not afrai...,0,2026-05-30T23:02:55Z,0


In [3]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'&amp;|&lt;|&gt;|&quot;', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

sample = df["text"].iloc[0]
print("Original:\n", sample)
print("\nCleaned:\n", clean_text(sample))

Original:
 Wow. It blows my mind the amount of commenters who thought that these videos were made by Ai or a software program. They've always been drawn by hand on an actual whiteboard with dry-erase markers. I would probably use Ai, but the smell of the markers is just too good. JK! There's almost nothing I love more than drawing - It's peaceful, fun and helps you learn a lot about yourself. The irony of Ai is that it's pushing many humans back into real world hobbies again - playing instruments, climbing trees, wood carving, welding, sports, reading physical books, etc. All this slop has made me appreciate the real world so much more.

Cleaned:
 wow it blows my mind the amount of commenters who thought that these videos were made by ai or a software program they ve always been drawn by hand on an actual whiteboard with dry erase markers i would probably use ai but the smell of the markers is just too good jk there s almost nothing i love more than drawing it s peaceful fun and helps 

In [4]:
df["text_clean"] = [clean_text(t) for t in tqdm(df["text"], desc="Cleaning")]

print("Done.")
print(df[["text", "text_clean"]].head(3))

Cleaning: 100%|██████████| 27330/27330 [00:00<00:00, 40333.28it/s]

Done.
                                                text  \
0  Wow. It blows my mind the amount of commenters...   
1  you can't create without destroying because yo...   
2  I'm madly in love of the way you are not afrai...   

                                          text_clean  
0  wow it blows my mind the amount of commenters ...  
1  you can t create without destroying because yo...  
2  i m madly in love of the way you are not afrai...  


In [5]:
SHORT_WHITELIST = {"ai", "ml"}

def get_lemmas(doc):
    return " ".join([
        token.lemma_.lower() for token in doc
        if not token.is_stop
        and not token.is_punct
        and not token.is_space
        and (len(token.text) >= 2 or token.lower_ in SHORT_WHITELIST)
    ])

# nlp.pipe processes in batches - much faster than per-row apply
texts = df["text_clean"].tolist()
lemmas = []

for doc in tqdm(nlp.pipe(texts, batch_size=256), total=len(texts), desc="Lemmatizing"):
    lemmas.append(get_lemmas(doc))

df["text_lemma"] = lemmas

print("Done.")
print("Sample:", df["text_lemma"].iloc[0])

Lemmatizing: 100%|██████████| 27330/27330 [01:38<00:00, 276.13it/s]

Done.
Sample: wow blow mind commenter think video ai software program ve draw hand actual whiteboard dry erase marker probably use ai smell marker good jk love draw peaceful fun help learn lot irony ai push human real world hobby play instrument climb tree wood carving weld sport read physical book etc slop appreciate real world


In [6]:
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/comments_preprocessed.csv", index=False, encoding="utf-8-sig")

print(f"Saved {len(df)} rows")
print("Columns:", df.columns.tolist())

Saved 27330 rows
Columns: ['video_id', 'video_name', 'comment_id', 'author', 'text', 'likes', 'published_at', 'reply_count', 'text_clean', 'text_lemma']
